In [ ]:
!pip install google-adk
!pip install requests python-dotenv

In [ ]:
import os

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-01-171e30612222"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
os.environ["MODEL"] = "gemini-2.0-flash"

print(f"Project: {os.environ['GOOGLE_CLOUD_PROJECT']}")
print(f"Location: {os.environ['GOOGLE_CLOUD_LOCATION']}")
print(f"Vertex AI: {os.environ['GOOGLE_GENAI_USE_VERTEXAI']}")

Project: qwiklabs-gcp-01-171e30612222
Location: us-central1
Vertex AI: True


In [ ]:
import logging
import asyncio
import sys
import requests
from typing import Optional, List, Dict

from google.adk.agents import LlmAgent, SequentialAgent, LoopAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.agent_tool import AgentTool
from google.genai import types
from google.api_core.exceptions import ResourceExhausted

try:
    from google.adk.tools import google_search
    search_tool = google_search
    print("Using ADK built-in google_search tool")
except ImportError:
    try:
        from google.adk.tools.google_search_tool import GoogleSearchTool
        search_tool = GoogleSearchTool()
        print("Using GoogleSearchTool class")
    except ImportError:
        from google.genai.types import Tool, GoogleSearch
        search_tool = Tool(google_search=GoogleSearch())
        print("Using Gemini native GoogleSearch grounding")

Using ADK built-in google_search tool


In [ ]:
logging.getLogger().handlers = []
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout
)
logger = logging.getLogger("challenge4")
logger.setLevel(logging.INFO)

print("Logging configured.")

Logging configured.


In [ ]:
def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            logger.info("[%s] USER >> %s", callback_context.agent_name, last.parts[0].text.strip()[:150])
    return None


def log_model_response(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            logger.info("[%s] MODEL >> %s", callback_context.agent_name, txt.strip()[:200])
    return None


print("Logging callbacks defined.")

Logging callbacks defined.


In [ ]:
def search_wikidata(query: str) -> str:
    """Searches Wikidata for structured geographic information about a place.

    Args:
        query: A search term describing a place, landmark, or geographic feature.

    Returns:
        A string containing structured data about matching places.
    """
    try:
        # Wikidata text search API — no key needed
        url = "https://www.wikidata.org/w/api.php"
        params = {
            "action": "wbsearchentities",
            "search": query,
            "language": "en",
            "format": "json",
            "limit": 5,
            "type": "item"
        }
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        results = []
        for item in data.get("search", []):
            label = item.get("label", "Unknown")
            desc = item.get("description", "No description")
            results.append(f"- {label}: {desc}")

        if results:
            return f"Wikidata results for '{query}':\n" + "\n".join(results)
        else:
            return f"No Wikidata results found for '{query}'."
    except Exception as e:
        return f"Wikidata search error: {str(e)}"


def search_places(query: str) -> str:
    """Searches OpenStreetMap Nominatim for places matching a description.

    Args:
        query: A place name or geographic description to search for.

    Returns:
        A string containing matching places with coordinates and types.
    """
    try:
        url = "https://nominatim.openstreetmap.org/search"
        params = {
            "q": query,
            "format": "json",
            "limit": 5,
            "addressdetails": 1
        }
        headers = {"User-Agent": "GeoGuesserBot/1.0"}
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()

        results = []
        for place in data:
            name = place.get("display_name", "Unknown")
            place_type = place.get("type", "unknown")
            lat = place.get("lat", "?")
            lon = place.get("lon", "?")
            results.append(f"- {name} (type: {place_type}, coords: {lat}, {lon})")

        if results:
            return f"Nominatim results for '{query}':\n" + "\n".join(results)
        else:
            return f"No Nominatim results found for '{query}'."
    except Exception as e:
        return f"Nominatim search error: {str(e)}"


def lookup_country_info(country_name: str) -> str:
    """Looks up detailed information about a country using REST Countries API.

    Args:
        country_name: The name of a country to look up.

    Returns:
        A string containing country details like languages, capital, region, population.
    """
    try:
        url = f"https://restcountries.com/v3.1/name/{country_name}"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()

        if not data:
            return f"No country data found for '{country_name}'."

        country = data[0]
        name = country.get("name", {}).get("common", "Unknown")
        capital = ", ".join(country.get("capital", ["Unknown"]))
        region = country.get("region", "Unknown")
        subregion = country.get("subregion", "Unknown")
        population = country.get("population", "Unknown")
        languages = ", ".join(country.get("languages", {}).values()) if country.get("languages") else "Unknown"
        currencies = ", ".join(
            f"{v.get('name', '?')} ({v.get('symbol', '?')})"
            for v in country.get("currencies", {}).values()
        ) if country.get("currencies") else "Unknown"
        timezones = ", ".join(country.get("timezones", ["Unknown"]))

        return (
            f"Country: {name}\n"
            f"Capital: {capital}\n"
            f"Region: {region} / {subregion}\n"
            f"Population: {population:,}\n"
            f"Languages: {languages}\n"
            f"Currencies: {currencies}\n"
            f"Timezones: {timezones}"
        )
    except Exception as e:
        return f"REST Countries error: {str(e)}"


# Quick tests
print("Geographic tools defined:")
print(f"  search_wikidata: {search_wikidata('Eiffel Tower')[:60]}...")
print(f"  search_places: {search_places('Pike Place Market')[:60]}...")
print(f"  lookup_country_info: {lookup_country_info('France')[:60]}...")

Geographic tools defined:
  search_wikidata: Wikidata search error: 403 Client Error: Forbidden for url: ...
  search_places: Nominatim results for 'Pike Place Market':
- Pike Place Mark...
  lookup_country_info: Country: France
Capital: Paris
Region: Europe / Western Euro...


In [ ]:
google_search_agent = LlmAgent(
    name="google_search_agent",
    model="gemini-2.0-flash",
    description="Searches Google for information about places, landmarks, and geographic features.",
    instruction="""You are a Google Search specialist. When given a query about
    a place or geographic feature, search Google and return the most relevant
    results. Focus on geographic, cultural, and landmark information.
    Return a concise summary of what you found.""",
    tools=[search_tool],  # google_search ONLY — no mixing
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

print("Google Search agent defined")

Google Search agent defined


In [ ]:
search_agent = LlmAgent(
    name="search_agent",
    model="gemini-2.0-flash",
    description="Researches locations using multiple knowledge sources.",
    instruction="""You are a geographic research specialist in a location guessing game.

    You have FOUR research tools:
    1. google_search_agent — ask it to search Google for location information
    2. search_wikidata — structured facts from Wikidata (landmarks, places)
    3. search_places — OpenStreetMap place search with coordinates
    4. lookup_country_info — country details (languages, capital, region)

    RESEARCH STRATEGY:
    1. Extract ALL clues from the description.
    2. Use google_search_agent for general web research on combined clues.
    3. Use search_wikidata for specific landmarks or features mentioned.
    4. Use search_places to verify candidate locations exist.
    5. Use lookup_country_info if language or cultural clues point to a country.

    IMPORTANT: If this is NOT the first round, read the previous critique
    and focus on the WEAKNESSES it identified.

    OUTPUT FORMAT:
    CLUES IDENTIFIED:
    - [every clue]

    RESEARCH FINDINGS:
    [what each tool returned]

    TOP CANDIDATES (ranked):
    1. [Location] — matches [X/Y] clues
    2. [Location] — matches [X/Y] clues
    3. [Location] — matches [X/Y] clues

    🌍 BEST GUESS: [Location, Country]
    CONFIDENCE: [HIGH / MEDIUM / LOW]""",
    tools=[
        AgentTool(agent=google_search_agent),
        search_wikidata,
        search_places,
        lookup_country_info,
    ],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

print("Search agent defined (4 knowledge sources)")

Search agent defined (4 knowledge sources)


In [ ]:
guess_agent = LlmAgent(
    name="guess_agent",
    model="gemini-2.0-flash",
    description="Makes a location guess by first researching, then reasoning about the clues.",
    instruction="""You are the location guesser. When given a description:

    1. First call the search_agent tool with the full description to get research.
    2. Based on the research, pick the SINGLE best location.
    3. Score every clue: ✅ matches, ❌ doesn't match, ❓ uncertain.
    4. Rate confidence: HIGH (80%+), MEDIUM (50-79%), LOW (<50%).
    5. If not HIGH, suggest what to investigate next.

    FORMAT:
    🌍 MY GUESS: [Location, Country]

    CLUE SCORECARD:
    ✅ [clue] — [evidence]
    ❌ [clue] — [evidence]

    SCORE: [X/Y] ([percentage]%)
    CONFIDENCE: [HIGH/MEDIUM/LOW]

    AREAS TO INVESTIGATE FURTHER:
    - [specific guidance for next round]""",
    tools=[AgentTool(agent=search_agent)],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

print("Guess agent defined (calls search_agent as tool)")

Guess agent defined (calls search_agent as tool)


In [ ]:
critique_agent = LlmAgent(
    name="critique_agent",
    model="gemini-2.0-flash",
    description="Evaluates a location guess by first getting the guess, then critiquing it.",
    instruction="""You are a geographic critic in a guessing game. When given a description:

    1. First call the guess_agent tool with the full description to get a guess.
    2. Evaluate the guess rigorously against ALL clues.
    3. Check if any alternative location would be a better fit.
    4. Assign a confidence rating.

    FORMAT:
    EVALUATING GUESS: [location from guess_agent]

    STRENGTHS:
    - [what matches well]

    WEAKNESSES:
    - [what doesn't match]

    ALTERNATIVE LOCATIONS:
    - [better candidate if any, with reasoning]

    CONFIDENCE: [HIGH/MEDIUM/LOW]

    VERDICT:
    If HIGH: "FINAL ANSWER: [location]. [Fun fact about the place]."
    If not HIGH: "NEEDS REFINEMENT: Focus next round on [specific guidance]." """,
    tools=[AgentTool(agent=guess_agent)],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

print("Critique agent defined (calls guess_agent as tool)")

Critique agent defined (calls guess_agent as tool)


In [ ]:
root_agent = LlmAgent(
    name="greeter",
    model="gemini-2.0-flash",
    description="GeoGuesser Bot — the game host.",
    instruction="""You are GeoGuesser Bot 🌍 — a location guessing game!

    WHEN THE USER DESCRIBES A PLACE:
    1. Say something enthusiastic
    2. Call the critique_agent tool with the user's full description
       (This triggers the full pipeline: search → guess → critique)
    3. If the critique says NEEDS REFINEMENT, call critique_agent AGAIN
       with the original description PLUS the refinement guidance.
       You may call it up to 5 times per response to refine the guess.
    4. Present the FINAL answer to the user
    5. Ask: "Did I get it right? 🎯"

    WHEN THE USER CONFIRMS (yes, correct):
    - Celebrate! "🎉 Got it!"

    WHEN THE USER SAYS NO / GIVES MORE CLUES:
    - Call critique_agent again with ALL clues combined
    - Ask the user for specific details if there are no guesses with high confidence, then guess again.
    - There is no limit to response cycles. The user can keep you guessing as long as they want.

    FOR GREETINGS:
    "Welcome to GeoGuesser Bot! 🌍 Describe any place and my team of
    researchers will investigate using Google Search, Wikidata,
    OpenStreetMap, and country databases. Type 'quit' to end."

    IMPORTANT: You are the host. Always present results yourself.
    Track how many times you've called critique_agent — stop at a max of 5 for each response.""",
    tools=[AgentTool(agent=critique_agent)],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

print("GeoGuesser Bot defined:")
print(f"  greeter (root)")
print(f"    → tool: critique_agent")
print(f"        → tool: guess_agent")
print(f"            → tool: search_agent")
print(f"                → tools: google_search, wikidata, nominatim, rest_countries")

GeoGuesser Bot defined:
  greeter (root)
    → tool: critique_agent
        → tool: guess_agent
            → tool: search_agent
                → tools: google_search, wikidata, nominatim, rest_countries


In [ ]:
async def run_single_query(query: str):
    """Sends a single query showing the agent chain."""
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")

    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name="geoguesser", user_id="test_user"
    )
    runner = Runner(
        agent=root_agent, app_name="geoguesser", session_service=session_service
    )
    user_message = types.Content(
        role="user", parts=[types.Part(text=query)]
    )

    max_retries = 3
    for attempt in range(max_retries):
        try:
            final_response = None
            events_log = []
            call_count = 0

            async for event in runner.run_async(
                user_id="test_user", session_id=session.id,
                new_message=user_message
            ):
                if hasattr(event, 'author') and event.author:
                    if event.author == "search_agent" and (
                        not events_log or events_log[-1] != "search_agent"
                    ):
                        call_count += 1
                        print(f"\n  --- ROUND {call_count} ---")
                    events_log.append(event.author)
                    print(f"  [EVENT] agent={event.author}")
                if event.is_final_response():
                    final_response = event

            print(f"\n  Research rounds: {call_count}")
            print(f"  Agent chain: {' → '.join(dict.fromkeys(events_log))}")

            if final_response and final_response.content and final_response.content.parts:
                print(f"\n{'─'*60}")
                print(f"FINAL RESPONSE:")
                print(f"{'─'*60}")
                print(final_response.content.parts[0].text)
            else:
                print("\nNo response received.")
            return final_response

        except ResourceExhausted:
            wait_time = 15 * (attempt + 1)
            print(f"\n[429] Waiting {wait_time}s... ({attempt+1}/{max_retries})")
            await asyncio.sleep(wait_time)
            if attempt == max_retries - 1:
                return None
        except Exception as e:
            print(f"\nError: {type(e).__name__}: {e}")
            return None

In [ ]:
result_1 = await run_single_query(
    "A place with a massive iron tower built for a world's fair, "
    "famous sidewalk cafes, a river, romantic language, incredible pastries."
)


QUERY: A place with a massive iron tower built for a world's fair, famous sidewalk cafes, a river, romantic language, incredible pastries.
22:38:35 [INFO] [greeter] USER >> A place with a massive iron tower built for a world's fair, famous sidewalk cafes, a river, romantic language, incredible pastries.
22:38:35 [INFO] Sending out request, model: gemini-2.0-flash, backend: GoogleLLMVariant.VERTEX_AI, stream: False
22:38:36 [INFO] Response received from the model.
22:38:36 [WARNING] Warning: there are non-text parts in the response: ['function_call'], returning concatenated text result from text parts. Check the full candidates.content.parts accessor to get the full model response.
22:38:36 [INFO] [greeter] MODEL >> Oh, how exciting! Let's find this place!
  [EVENT] agent=greeter
22:38:36 [INFO] [critique_agent] USER >> A place with a massive iron tower built for a world's fair, famous sidewalk cafes, a river, romantic language, incredible pastries.
22:38:36 [INFO] Sending out request,

In [ ]:
result_2 = await run_single_query(
    "Red desert, ancient sacred rock formations, extremely remote, "
    "incredible stars, hot days cold nights, indigenous culture tens "
    "of thousands of years old."
)


QUERY: Red desert, ancient sacred rock formations, extremely remote, incredible stars, hot days cold nights, indigenous culture tens of thousands of years old.
22:38:50 [INFO] [greeter] USER >> Red desert, ancient sacred rock formations, extremely remote, incredible stars, hot days cold nights, indigenous culture tens of thousands of years ol
22:38:50 [INFO] Sending out request, model: gemini-2.0-flash, backend: GoogleLLMVariant.VERTEX_AI, stream: False
22:38:51 [INFO] Response received from the model.
22:38:51 [INFO] [greeter] MODEL >> Wow, that sounds incredible! 🤩 Let me pinpoint that for you.
  [EVENT] agent=greeter
22:38:51 [INFO] [critique_agent] USER >> Red desert, ancient sacred rock formations, extremely remote, incredible stars, hot days cold nights, indigenous culture tens of thousands of years ol
22:38:51 [INFO] Sending out request, model: gemini-2.0-flash, backend: GoogleLLMVariant.VERTEX_AI, stream: False
22:38:53 [INFO] Response received from the model.
22:38:53 [INFO] 

In [ ]:
result_3 = await run_single_query(
    "Coastal city with colorful houses on hills, street art, famous "
    "custard tarts, trams in narrow streets, a large red bridge "
    "that isn't the Golden Gate."
)


QUERY: Coastal city with colorful houses on hills, street art, famous custard tarts, trams in narrow streets, a large red bridge that isn't the Golden Gate.
22:38:54 [INFO] [greeter] USER >> Coastal city with colorful houses on hills, street art, famous custard tarts, trams in narrow streets, a large red bridge that isn't the Golden Gate.
22:38:54 [INFO] Sending out request, model: gemini-2.0-flash, backend: GoogleLLMVariant.VERTEX_AI, stream: False
22:38:55 [INFO] Response received from the model.
22:38:55 [INFO] [greeter] MODEL >> That sounds like an amazing place! Let me see if I can pinpoint it.
  [EVENT] agent=greeter
22:38:55 [INFO] [critique_agent] USER >> Coastal city with colorful houses on hills, street art, famous custard tarts, trams in narrow streets, a large red bridge that isn't the Golden Gate.
22:38:55 [INFO] Sending out request, model: gemini-2.0-flash, backend: GoogleLLMVariant.VERTEX_AI, stream: False
22:38:56 [INFO] Response received from the model.
22:38:56 [INFO

In [ ]:
result_4 = await run_single_query("Hello! How does this game work?")


QUERY: Hello! How does this game work?
22:39:10 [INFO] [greeter] USER >> Hello! How does this game work?
22:39:10 [INFO] Sending out request, model: gemini-2.0-flash, backend: GoogleLLMVariant.VERTEX_AI, stream: False
22:39:11 [INFO] Response received from the model.
22:39:11 [INFO] [greeter] MODEL >> Welcome to GeoGuesser Bot! 🌍 Describe any place and my team of
researchers will investigate using Google Search, Wikidata,
OpenStreetMap, and country databases. Type 'quit' to end.
  [EVENT] agent=greeter

  Research rounds: 0
  Agent chain: greeter

────────────────────────────────────────────────────────────
FINAL RESPONSE:
────────────────────────────────────────────────────────────
Welcome to GeoGuesser Bot! 🌍 Describe any place and my team of
researchers will investigate using Google Search, Wikidata,
OpenStreetMap, and country databases. Type 'quit' to end.



In [ ]:
def check_result(result, label):
    if result is None:
        return f"  [????] {label}: No result captured (rate limited?)"
    text = ""
    if result.content and result.content.parts:
        text = result.content.parts[0].text.lower()
    has_content = len(text) > 50 and "error" not in text
    status = "PASS" if has_content else "FAIL"
    return f"  [{status}] {label}: {'Substantive response' if has_content else 'No substantive response'}"


print("""
╔══════════════════════════════════════════════════════════════════╗
║           CHALLENGE 4 — REQUIREMENTS VALIDATION                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  REQUIREMENT: Agent workflow that verifies and refines answers   ║
║  ─────────────────────────────────────────────────────────────   ║
║  Implementation: Chained AgentTools (no sub_agents needed)       ║
║  Theme: GeoGuesser with 4 knowledge sources                      ║
║                                                                  ║
║  AGENT CHAIN (each calls the next as a tool):                    ║
║    greeter (root)                                                ║
║      → critique_agent  (evaluates + decides to refine)           ║
║          → guess_agent  (makes guess with scorecard)             ║
║              → search_agent (4 knowledge sources)                ║
║                  → google_search                                 ║
║                  → wikidata                                      ║
║                  → nominatim/OSM                                 ║
║                  → REST Countries                                ║
║                                                                  ║
║  REFINEMENT: Root calls critique_agent up to 3 times.            ║
║  Each call triggers the full chain. If critique says             ║
║  "NEEDS REFINEMENT", root calls again with guidance.             ║
║  Look for multiple --- ROUND N --- markers.                      ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

print("FUNCTIONAL CHECKS:")
print("=" * 60)
print()
print("Location guesses:")
print(check_result(result_1, "Easy: Paris"))
print(check_result(result_2, "Medium: Uluru"))
print(check_result(result_3, "Hard: Lisbon"))
print()
print("Direct interaction:")
print(check_result(result_4, "Greeting"))

all_results = [result_1, result_2, result_3, result_4]
passed = sum(1 for r in all_results if check_result(r, "").strip().startswith("[PASS]"))
print(f"\nRESULT: {passed}/{len(all_results)} checks passed")
if passed == len(all_results):
    print("All checks passed.")


╔══════════════════════════════════════════════════════════════════╗
║           CHALLENGE 4 — REQUIREMENTS VALIDATION                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  REQUIREMENT: Agent workflow that verifies and refines answers    ║
║  ─────────────────────────────────────────────────────────────   ║
║  Implementation: Chained AgentTools (no sub_agents needed)       ║
║  Theme: GeoGuesser with 4 knowledge sources                      ║
║                                                                  ║
║  AGENT CHAIN (each calls the next as a tool):                    ║
║    greeter (root)                                                ║
║      → critique_agent  (evaluates + decides to refine)           ║
║          → guess_agent  (makes guess with scorecard)             ║
║              → search_agent (4 knowledge sources)                ║
║                  → google_sear

In [ ]:
EXIT_COMMANDS = {"quit", "exit", "stop", "end", "bye", "done", "q"}

async def play_geoguesser():
    print("=" * 60)
    print("🌍 GEOGUESSER BOT — INTERACTIVE MODE")
    print("=" * 60)
    print("Describe a place and I'll guess it!")
    print("Type 'quit' to exit, 'new' for fresh session.\n")

    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name="geoguesser", user_id="player"
    )
    runner = Runner(
        agent=root_agent, app_name="geoguesser", session_service=session_service
    )

    while True:
        try:
            user_input = input("\n🎮 YOU: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Thanks for playing!")
            return
        if not user_input:
            continue
        if user_input.lower() in EXIT_COMMANDS:
            print("\n👋 Thanks for playing GeoGuesser Bot!")
            return
        if user_input.lower() in {"new", "new game", "reset"}:
            session = await session_service.create_session(
                app_name="geoguesser", user_id="player"
            )
            print("🔄 New game! Describe a place.")
            continue

        user_message = types.Content(
            role="user", parts=[types.Part(text=user_input)]
        )
        max_retries = 3
        for attempt in range(max_retries):
            try:
                final_response = None
                round_count = 0
                async for event in runner.run_async(
                    user_id="player", session_id=session.id,
                    new_message=user_message
                ):
                    if hasattr(event, 'author') and event.author == "search_agent":
                        round_count += 1
                        print(f"  🔍 Research round {round_count}...")
                    if event.is_final_response():
                        final_response = event
                if final_response and final_response.content and final_response.content.parts:
                    print(f"\n🤖 GEOGUESSER: {final_response.content.parts[0].text}")
                else:
                    print("\n🤖 GEOGUESSER: Hmm, more clues please?")
                break
            except ResourceExhausted:
                wait_time = 15 * (attempt + 1)
                print(f"  ⏳ Rate limited. Waiting {wait_time}s...")
                await asyncio.sleep(wait_time)
            except Exception as e:
                print(f"  ❌ Error: {e}")
                break

await play_geoguesser()

🌍 GEOGUESSER BOT — INTERACTIVE MODE
Describe a place and I'll guess it!
Type 'quit' to exit, 'new' for fresh session.


🎮 YOU: It's a city. It has a coast. I has a long park with a large strange shiny thing in it.
22:40:02 [INFO] [greeter] USER >> It's a city. It has a coast. I has a long park with a large strange shiny thing in it.
22:40:02 [INFO] Sending out request, model: gemini-2.0-flash, backend: GoogleLLMVariant.VERTEX_AI, stream: False
22:40:03 [INFO] Response received from the model.
22:40:03 [INFO] [greeter] MODEL >> Oh, that sounds intriguing! Let's see...
22:40:03 [INFO] [critique_agent] USER >> It's a city. It has a coast. I has a long park with a large strange shiny thing in it.
22:40:03 [INFO] Sending out request, model: gemini-2.0-flash, backend: GoogleLLMVariant.VERTEX_AI, stream: False
22:40:04 [INFO] Response received from the model.
22:40:04 [INFO] [guess_agent] USER >> It's a city. It has a coast. I has a long park with a large strange shiny thing in it.
22:40:04 [